# Portion 2 · TabBERT encoder: pretraining and handoffs

Pretrain the two-level TabBERT encoder with masked field modelling on train-period windows, then export embeddings, fine-tuning scores and reconstruction scores for Portions 3 and 4. Design: [`PORTION2_BUILD_GUIDE.md`](PORTION2_BUILD_GUIDE.md). Step-by-step plan: [`PORTION2_IMPLEMENTATION_PLAN.md`](PORTION2_IMPLEMENTATION_PLAN.md).

Run on a Colab GPU runtime (Runtime → Change runtime type → T4 GPU or better). Checkpoints and outputs go to Google Drive, not to Git.

## 1. Get the code and the Portion 1 data

The first code cell clones the repository and pulls only the Portion 1 files this portion reads through Git LFS (about 190 MB instead of the full 530 MB), which keeps LFS bandwidth low. When the notebook runs from a local checkout, it uses that checkout instead.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/ted-cc6/TabBERT_AML"
BRANCH = "main"
LFS_FILES = [
    "Portion1/processed/event_tokens.npz",
    "Portion1/processed/window_index.csv.gz",
    "Portion1/processed/events_meta.csv.gz",
    "Portion1/processed/label_subsets.csv.gz",
    "Portion1/processed/label_subset_100pct.csv.gz",
]

current = Path.cwd()
PORTION2_DIR = next((p for p in (current, current / "Portion2") if (p / "portion2_encoder.py").is_file()), None)

if PORTION2_DIR is None:
    clone_root = Path("/content/aml_project_repo")
    if shutil.which("git-lfs") is None:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "git-lfs"], check=True)
    subprocess.run(["git", "lfs", "install", "--skip-smudge"], check=True)
    if not clone_root.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(clone_root)], check=True)
    else:
        subprocess.run(["git", "-C", str(clone_root), "pull", "--ff-only"], check=True)
    subprocess.run(["git", "-C", str(clone_root), "lfs", "pull", "--include", ",".join(LFS_FILES)], check=True)
    PORTION2_DIR = clone_root / "Portion2"

PROCESSED_DIR = PORTION2_DIR.parent / "Portion1" / "processed"
sys.path.insert(0, str(PORTION2_DIR))
for name in LFS_FILES:
    size = (PORTION2_DIR.parent / name).stat().st_size
    assert size > 1_000_000, f"{name} is still an LFS pointer ({size} bytes)"
print(PORTION2_DIR)

## 2. Check the GPU and mount Drive

`OUT_DIR` is the shared Drive folder for checkpoints, metrics, embeddings and score files. Each run writes to `OUT_DIR/<run_name>/`.

In [ ]:
import torch

import portion2_encoder as enc

DEVICE = enc.pick_device()
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
if DEVICE.type != "cuda":
    print("No CUDA GPU: switch the runtime to a GPU before the full run.")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/TabBERT_AML/portion2_runs")
except ImportError:
    OUT_DIR = PORTION2_DIR / "runs"  # local runs; ignored by Git
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs:", OUT_DIR)

## 3. Main pretraining, seed 42 (Task 4)

Field-level masking with Δt, 1M train windows per epoch, 3 epochs, batch 512, AdamW 1e-3 with warmup and cosine decay. Progress prints every 200 steps. Each run writes `checkpoint.pt`, `metrics.csv` and `amount_embedding_cosine.png` to `OUT_DIR/<run_name>/`.

`run_or_load` skips training when the run's checkpoint is already on Drive, so the notebook can be rerun from the top without retraining. Set `QUICK_CHECK = True` to test a new runtime on 50K windows (about a minute).

In [ ]:
import dataclasses
import time

QUICK_CHECK = False


def run_or_load(config_name, **overrides):
    cfg = enc.load_run_config(PORTION2_DIR / "configs" / f"{config_name}.json")
    if overrides:
        cfg = dataclasses.replace(cfg, **overrides)
    ckpt = OUT_DIR / cfg.run_name / "checkpoint.pt"
    if ckpt.exists():
        print("Already trained:", ckpt)
        return ckpt
    start = time.time()
    ckpt = enc.pretrain(cfg, PROCESSED_DIR, OUT_DIR, device=DEVICE, log_every=200)
    print(f"Done in {(time.time() - start) / 60:.1f} min:", ckpt)
    return ckpt


if QUICK_CHECK:
    main_ckpt = run_or_load("main_seed42", run_name="quick_check", windows_per_epoch=50_000, epochs=1,
                            valid_windows=5_000, warmup_steps=20)
else:
    main_ckpt = run_or_load("main_seed42")

## 4. Check the result

Done when each field's masked accuracy (`acc_`) is clearly above its most-frequent-token baseline (`baseline_`), especially for currency, counterparty bank, amount and time gap. Direction at the last position is excluded, because the target event is always "sent". The heatmap should show neighbouring amount buckets as similar (a bright band along the diagonal).

In [ ]:
import pandas as pd
from IPython.display import Image, display


def summarize(ckpt_path):
    metrics = pd.read_csv(ckpt_path.parent / "metrics.csv")
    _, specs, _ = enc.load_encoder(ckpt_path, torch.device("cpu"))
    fields = [s.name for s in specs]
    summary = pd.DataFrame({
        "accuracy": [metrics[f"acc_{f}"].iloc[-1] for f in fields],
        "baseline": [metrics[f"baseline_{f}"].iloc[-1] for f in fields],
    }, index=fields)
    summary["lift"] = summary.accuracy - summary.baseline
    extra = [c for c in metrics.columns if c.startswith("acc1_") or c.endswith("_nonzero")]
    print(metrics[["epoch", "train_loss", "valid_loss", "seconds"]].to_string(index=False), "\n")
    print(summary.round(3).to_string(), "\n")
    print(metrics[["epoch"] + extra].round(3).to_string(index=False))
    display(Image(filename=str(ckpt_path.parent / "amount_embedding_cosine.png")))


summarize(main_ckpt)

## 5. Larger-model check

One extra pretraining run with a model about 3 times larger (4.3M parameters: 4 sequence layers, d_model 256, feed-forward 1024), same data, seed and schedule. Rule, fixed before the run: keep the larger model only if its final validation masked loss is at least 5% lower than the main model's. Otherwise keep the 1.5M model for all checkpoints and downstream runs. Only validation data is used for this choice.

In [ ]:
capacity_ckpt = run_or_load("capacity_check_seed42")


def compare(ckpts):
    table = {}
    for ckpt in ckpts:
        model, specs, cfg = enc.load_encoder(ckpt, torch.device("cpu"))
        last = pd.read_csv(ckpt.parent / "metrics.csv").iloc[-1]
        table[cfg.run_name] = {
            "parameters": sum(p.numel() for p in model.parameters()),
            "valid_loss": last.valid_loss,
            "train_loss": last.train_loss,
            "minutes": last.seconds / 60,
            **{f"acc_{s.name}": last[f"acc_{s.name}"] for s in specs},
        }
    return pd.DataFrame(table)


comparison = compare([main_ckpt, capacity_ckpt])
print(comparison.round(4).to_string())
gain = 1 - comparison.loc["valid_loss"].iloc[1] / comparison.loc["valid_loss"].iloc[0]
print(f"\nValidation loss reduction from the larger model: {gain:.1%} (rule: keep it only if >= 5%)")

## 6. Export frozen embeddings (Task 5)

One 256-value embedding per transaction for Portion 3: the target-row vector (first 128 columns), then the mean over the window's real rows (last 128). If Portion 3 wants 128 values, they take the first 128 columns. Rows follow `window_index.csv.gz` order and cover all splits including tail; the `_ids.npy` file holds the `transaction_id` of each row, so join on that, not on row order of other files.

The file is about 2.6 GB. It is written to the runtime's local disk first, then copied to Drive (writing a memory-mapped file straight to Drive is slow).

In [ ]:
import numpy as np

EXPORT_RUN = "main_seed42"
local_dir = Path("/content/exports") if Path("/content").exists() else OUT_DIR / "exports"
emb_path, ids_path = enc.export_embeddings(OUT_DIR / EXPORT_RUN / "checkpoint.pt", PROCESSED_DIR, local_dir, device=DEVICE)

target_dir = OUT_DIR / EXPORT_RUN
if local_dir != target_dir:
    for path in (emb_path, ids_path):
        shutil.copyfile(path, target_dir / path.name)  # copy2 also copies timestamps, which some mounts refuse
        print("Copied to", target_dir / path.name)

embeddings = np.load(target_dir / emb_path.name, mmap_mode="r")
ids = np.load(target_dir / ids_path.name)
print(embeddings.shape, embeddings.dtype, f"{embeddings.nbytes / 1e9:.2f} GB")
print("ids:", len(ids), "unique:", len(np.unique(ids)))
print("finite in first 100K rows:", bool(np.isfinite(embeddings[:100_000]).all()))